# 10 Model F: LoRA inside CLIP's last blocks (F1-1, F1-4, F2)

Runs `scripts/run_model_f.py` (plan: CLAUDE.md scope + the proposal). It downloads the pinned UTKFace archive and CLIP ViT-L/14 weights, stores the tokens entering block 20 once (~12.5 GB on the runtime's local disk), rebuilds model A per fold and trains F with the same validation-fold rule as E. **Step 1 times one fold first**; run step 2 only when that time fits your budget. GPU: T4 or better. No images are displayed.

In [ ]:
import os, pathlib, subprocess, json, shutil
os.chdir('/content')
REPO = pathlib.Path('/content/robust_child_filtering_model')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/isaacmutuma/robust_child_filtering_model.git', str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1
!pip -q install open_clip_torch==3.3.0
WORK = '/content/work'
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Time one fold of F1-4 (the slowest F1 setting)

In [ ]:
!python scripts/run_model_f.py --work {WORK} --configs F1_4 --folds 0

## 2. Full runs (5 folds each): F1-1, F1-4, then F2 (uses the better F1 depth on the validation folds)

In [ ]:
!python scripts/run_model_f.py --work {WORK} --configs F1_1,F1_4,F2

## 3. Save the aggregate results to Drive (JSON only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
dst = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model/results/models'); dst.mkdir(parents=True, exist_ok=True)
for n in ['F1_1', 'F1_4', 'F2']:
    src = REPO / 'results' / 'models' / f'{n}.json'
    if src.exists():
        d = json.loads(src.read_text()); assert d['git']['dirty'] is False, 'code was modified'
        shutil.copy(src, dst / src.name); print('saved', src.name, 'commit', d['git']['commit'][:7])